<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Dia_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Dia 1.6B — Ultra-Realistic Dialogue with Non-Verbal Sounds

1.6B-parameter open-weights text-to-dialogue model by Nari Labs. Generates highly realistic multi-speaker conversation in one pass, with **21 non-verbal tags** for laughs, coughs, sighs, gasps, and more. Supports voice cloning from a 5–10 second reference clip. English only.

### Quick Start
1. Connect to a GPU runtime (T4 16 GB works, L4/A100 recommended for speed)
2. Run Steps 1–4 in order — no token, no sign-up needed
3. Open the Gradio UI link from Step 4 and start generating

| GPU | VRAM | Speed (no compile) |
|-----|------|-------------------|
|-----|------|-------------------|
|-----|------|-------------------|
| A100 | 40 GB | Fastest |
|-----|------|-------------------|
| L4 | 24 GB | Fast |
|-----|------|-------------------|
| T4 | 16 GB | Works in bf16 (~4.4 GB VRAM), slower |

**First run also downloads the Descript Audio Codec (DAC)** for decoding audio prompts.

### Generation Tips (from the model authors)
- **Keep input length moderate** — under 5s of audio is unnatural, over 20s comes out unnaturally fast.
- **Always start with `[S1]`** and alternate between `[S1]` and `[S2]`. Don't do `[S1]... [S1]...` consecutively.
- **Use non-verbal tags sparingly** — see the list below; overusing or using unlisted ones can cause artifacts.
- **Voice cloning** — upload a 5–10s reference clip, provide its transcript using `[S1]` / `[S2]` tags correctly, then put your new script after. Use the second-to-last speaker's tag at the end to improve tail quality.

### Non-Verbal Tags
`<laughs>`, `<clears throat>`, `<sighs>`, `<gasps>`, `<coughs>`, `<singing>`, `<sings>`, `<mumbles>`, `<beep>`, `<groans>`, `<sniffs>`, `<claps>`, `<screams>`, `<inhales>`, `<exhales>`, `<applause>`, `<burps>`, `<humming>`, `<sneezes>`, `<chuckle>`, `<whistles>`

**License:** Apache 2.0. The model card explicitly forbids using the model for identity misuse (cloning real people without permission), deceptive content, and illegal or malicious use.

In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Installs the official `nari-tts` (Dia) package via git, plus the Descript Audio Codec and Gradio.

import os, sys, subprocess
import torch

if not torch.cuda.is_available():
    raise SystemExit('No GPU detected. Connect a GPU runtime: Runtime \u2192 Change runtime type \u2192 T4 / L4 / A100')

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'[GPU] {gpu_name} — {vram_gb:.1f} GB')

if vram_gb < 10:
    raise SystemExit(
        f'\u274c {vram_gb:.0f} GB VRAM is not enough. Dia 1.6B in bf16 needs ~4.4 GB, '
        'plus headroom for the Descript Audio Codec and KV cache. '
        'Reconnect to a runtime with \u226516 GB (T4 or better).'
    )

print('[pip] Installing nari-tts (Dia) from GitHub ...')
# --no-deps prevents torchaudio upgrade (Colab torch 2.11 ABI mismatch)
subprocess.run(['pip', 'install', '-q', '-U', '--no-deps', 'git+https://github.com/nari-labs/dia.git'], check=True)
# Install Dia's actual runtime deps from pyproject.toml (skip torch/torchaudio — Colab ships them)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'descript-audio-codec>=1.0.0', 'tokenizers==0.21.0', 'einops', 'pydantic', 'safetensors'], check=True)
try:
    import dia
    print(f'  nari-tts installed (dia package, version unknown)')
except ImportError as e:
    raise SystemExit(f'\u274c Failed to import dia: {e}')

print('[pip] Installing Gradio UI extras ...')
subprocess.run(['pip', 'install', '-q', '-U', 'gradio==5.33.0', 'soundfile', 'librosa', 'numpy', 'tqdm==4.66.5'], check=True)

os.makedirs('/content/audio_out', exist_ok=True)
os.makedirs('/content/ref_audio', exist_ok=True)
print('[Setup] /content/audio_out and /content/ref_audio ready.')

In [ ]:
# @title STEP 2 — Pre-cache Models
# @markdown Downloads the 1.6B weights (bf16, ~3.2 GB) and the Descript Audio Codec to the local cache.

# @markdown Reuses the Drive-cached weights from `TTS_Model_Loader.ipynb` if present,
# @markdown otherwise downloads to the default ~/.cache/huggingface (in-session only).
import pathlib
try:
    from google.colab import drive
    if not pathlib.Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive', force_remount=False)
    CACHE_ROOT = pathlib.Path('/content/drive/MyDrive/AEI_TTS_Cache')
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ.setdefault('HF_HOME', str(CACHE_ROOT / 'huggingface'))
    print(f'[Cache] using Drive at {CACHE_ROOT}')
except Exception:
    print('[Cache] Drive not available, using default ~/.cache/huggingface')


import os
from huggingface_hub import snapshot_download

REPO = 'nari-labs/Dia-1.6B-0626'
print(f'[Download] {REPO} (~3.2 GB) ...')
snapshot_download(REPO)
print('[Download] Dia weights cached. The DAC codec downloads on first model load.')

In [ ]:
# @title STEP 3 — Core Functions (lazy model loading)

# @markdown Wraps the `dia.model.Dia` class. Model is loaded on first use (a few minutes for the DAC codec).



import os, time

import numpy as np

import torch

import soundfile as sf



REPO = 'nari-labs/Dia-1.6B-0626'

_MODEL = None



def get_model():

    global _MODEL

    if _MODEL is not None:

        return _MODEL

    from dia.model import Dia

    print(f'[Load] {REPO} (bf16) ...')

    t0 = time.time()

    _MODEL = Dia.from_pretrained(REPO, compute_dtype='bfloat16')

    print(f'[Load] Ready in {time.time()-t0:.1f}s')

    return _MODEL



def synth(text: str,

          audio_prompt_path: str | None = None,

          max_tokens: int = 3072,

          cfg_scale: float = 3.0,

          temperature: float = 1.8,

          top_p: float = 0.90,

          top_k: int = 45,

          cfg_filter_top_k: int = 50,

          seed: int | None = None,

          out_name: str = 'dia.wav') -> tuple[str, int]:

    """Run a single Dia generation. Returns (path, sample_rate)."""

    m = get_model()

    text = (text or '').strip()

    if not text:

        raise RuntimeError('Text is empty.')

    if not text.startswith('[S'):

        # Dia requires a speaker tag at the start

        text = '[S1] ' + text



    kwargs = dict(

        text=text,

        max_new_tokens=int(max_tokens),

        cfg_scale=float(cfg_scale),

        temperature=float(temperature),

        top_p=float(top_p),

        cfg_filter_top_k=int(cfg_filter_top_k),

        top_k=int(top_k),

        use_torch_compile=False,  # skip torch.compile (slow first run, often OOMs in Colab)

        verbose=True,

    )

    if audio_prompt_path:

        kwargs['audio_prompt'] = audio_prompt_path

    if seed is not None and int(seed) >= 0:

        torch.manual_seed(int(seed))



    output = m.generate(**kwargs)

    out_path = os.path.join('/content/audio_out', out_name)

    m.save_audio(out_path, output)

    # Dia's save_audio writes a wav; sample rate is 44100 per the model card

    return out_path, 44100



print('[Core] Ready. Use Step 4 for the UI, Step 6 for quick test, Step 7 for batch.')

In [ ]:
# @title STEP 4 — Gradio UI
# @markdown Interactive web app. Type dialogue with `[S1]` and `[S2]` tags, optionally upload a reference clip for voice cloning. Click the `.gradio.live` link to open.

import sys
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass
import gradio as gr

def _err(msg):
    return None, f'\u274c {msg}'

def ui_synth(text, audio_prompt, max_tokens, cfg_scale, temperature, top_p, top_k, seed, progress=gr.Progress()):
    if not text or not text.strip():
        return _err('Type some dialogue first.')
    try:
        progress(0.05, desc='Loading Dia (first run takes a few minutes)...')
        get_model()
        progress(0.15, desc='Generating...')
        path, sr = synth(
            text.strip(),
            audio_prompt_path=audio_prompt,
            max_tokens=max_tokens,
            cfg_scale=cfg_scale,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            seed=int(seed) if seed is not None and int(seed) >= 0 else None,
            out_name='ui.wav',
        )
        progress(1.0, desc='Done.')
        return path, f'\u2705 Generated @ {sr} Hz'
    except Exception as e:
        return None, f'\u274c {e}'

DEFAULT_TEXT = "[S1] Dia is an open weights text to dialogue model. [S2] You get full control over scripts and voices. [S1] Wow. Amazing. (laughs) [S2] Try it now on Git hub or Hugging Face."

with gr.Blocks(title='Dia 1.6B', theme=gr.themes.Soft()) as demo:
    gr.Markdown(f'# \ud83d\udc4b Dia 1.6B\n**Model:** `nari-labs/Dia-1.6B-0626` (1.6B, bf16) \u00b7 **GPU:** {gpu_name} ({vram_gb:.0f} GB)\n\nUltra-realistic multi-speaker dialogue with 21 non-verbal tags. Voice cloning supported via 5–10s reference clips.')
    with gr.Row():
        with gr.Column():
            text = gr.Textbox(
                label='Dialogue (use [S1] / [S2] to switch speakers, non-verbals like (laughs) are honored)',
                value=DEFAULT_TEXT,
                lines=5,
                placeholder='[S1] Hello there. [S2] Hi! (laughs) [S1] What\'s so funny?',
            )
            with gr.Accordion('Voice cloning (optional)', open=False):
                audio_prompt = gr.Audio(
                    label='Reference clip (5\u201310 seconds, [S1]/[S2]-tagged transcript must appear at the START of the dialogue above)',
                    type='filepath', info="5-10s of multi-speaker dialogue. Transcript must be the FIRST part of the dialogue above (with [S1]/[S2] tags).",
                    )
                gr.Markdown('\u26a0\ufe0f The transcript of your reference clip must be the first part of the dialogue above, properly tagged with `[S1]` and `[S2]`. Only the audio matching the script AFTER the reference transcript is returned.')
            with gr.Accordion('Advanced sampling', open=False):
                max_tokens = gr.Slider(512, 4096, value=3072, step=64, label='Max new tokens (86 tokens \u2248 1 second)', info="Maximum tokens to generate. 86 tokens ≈ 1 second of audio.")
                cfg_scale = gr.Slider(1.0, 6.0, value=3.0, step=0.1, label='CFG scale (higher = stricter adherence to script)', info="Classifier-free guidance: higher = more strictly follows the script.")
                temperature = gr.Slider(0.5, 2.5, value=1.8, step=0.05, label='Temperature', info="Lower = more deterministic, higher = more varied. 0.7 is a good default.")
                top_p = gr.Slider(0.1, 1.0, value=0.9, step=0.01, label='Top-p', info="Nucleus sampling: only consider tokens whose cumulative probability is below this. 0.9 = off.")
                top_k = gr.Slider(1, 200, value=45, step=1, label='Top-k', info="Only consider top K tokens at each step. 50 is a good default; 0 = disabled.")
                seed = gr.Number(value=-1, label='Seed (-1 = random)', precision=0)
            btn = gr.Button('Generate dialogue', variant='primary', size='lg')
        with gr.Column():
            output_audio = gr.Audio(label='Generated dialogue', type='filepath')
            status = gr.Markdown('')
    gr.Examples(
        examples=[
            ["[S1] Hey, did you see the news? [S2] No, what happened? [S1] (clears throat) They released an open-weights dialogue model. [S2] (gasps) No way! (laughs) That's amazing!", None],
            ["[S1] I think I have a cold. [S2] (sniffs) Aw, you sound terrible. [S1] (coughs) Thanks. [S2] (chuckle) You're welcome.", None],
            ["[S1] (singing) Happy birthday to you. [S2] (singing) Happy birthday to you. [S1] (singing) Happy birthday dear friend. [S2] (applause)", None],
        ],
        inputs=[text, audio_prompt],
    )
    gr.Markdown('**Tips:** Keep the script under 20 seconds of audio or it comes out unnaturally fast. Always alternate `[S1]` and `[S2]`. The model is English only. Different seeds will produce different voices; use a reference clip (voice cloning) for consistency.')
    btn.click(
        ui_synth,
        inputs=[text, audio_prompt, max_tokens, cfg_scale, temperature, top_p, top_k, seed],
        outputs=[output_audio, status],
    )

from IPython.display import clear_output as _clear
_clear()
demo.queue(max_size=8, concurrency_limit=2).launch(
    share=True, show_error=True, server_name="0.0.0.0", server_port=7860,
)
demo.load(lambda: "🎙️ Dia 1.6B ready. Add [S1]/[S2] tags for multi-speaker dialogue. See the tips below.", outputs=[status])


In [ ]:
# @title Step 5 — Keep Alive
# @markdown Prevents Colab from disconnecting during long generation sessions. Run anytime after launching the UI.

import threading, time
def _keep():
    while True:
        time.sleep(60)
        try:
            import requests
            requests.get('https://www.google.com', timeout=5)
        except Exception:
            pass
threading.Thread(target=_keep, daemon=True).start()
print('[Keep-Alive] Running. Stop cell to disable.')

In [ ]:
# @title Step 6 — Quick Test (one-off inference, no UI)
# @markdown Run a single dialogue generation from the notebook. Useful for scripting or quick checks.

TEXT = "[S1] Dia is an open weights text to dialogue model. [S2] You get full control over scripts and voices. [S1] Wow. Amazing. (laughs) [S2] Try it now on Git hub or Hugging Face." # @param {type:"string"}
AUDIO_PROMPT = "" # @param {type:"string"}
MAX_TOKENS = 3072 # @param {type:"integer"}
CFG_SCALE = 3.0 # @param {type:"number"}
TEMPERATURE = 1.8 # @param {type:"number"}
TOP_P = 0.9 # @param {type:"number"}
TOP_K = 45 # @param {type:"integer"}
SEED = -1 # @param {type:"integer"}

path, sr = synth(
    TEXT,
    audio_prompt_path=AUDIO_PROMPT or None,
    max_tokens=MAX_TOKENS,
    cfg_scale=CFG_SCALE,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    seed=SEED,
)
print(f'[Done] {path} ({sr} Hz)')
from IPython.display import Audio, display
display(Audio(path))

In [ ]:
# @title Step 7 — Batch Synthesis
# @markdown Generate audio for a list of dialogue scripts. Each line in `BATCH` becomes one output file. **One script per line — multi-speaker lines should be combined into a single string with `[S1]... [S2]...` tags.**

BATCH_AUDIO_PROMPT = '' # @param {type:"string"}
BATCH_MAX_TOKENS = 3072 # @param {type:"integer"}
BATCH_CFG_SCALE = 3.0 # @param {type:"number"}
BATCH_TEMPERATURE = 1.8 # @param {type:"number"}
BATCH_TOP_P = 0.9 # @param {type:"number"}
BATCH_TOP_K = 45 # @param {type:"integer"}
BATCH_SEED = -1 # @param {type:"integer"}

BATCH = """\
[S1] The quick brown fox jumps over the lazy dog. [S2] That\'s a classic.
[S1] To be, or not to be: that is the question. [S2] (sighs) Heavy.
[S1] Knock knock. [S2] Who\'s there? [S1] Cow says. [S2] Cow says who? [S1] (laughs) No, a cow says moo!
[S1] I just got promoted at work! [S2] (gasps) That\'s incredible! Congratulations!
"""

lines = [l.strip() for l in BATCH.strip().splitlines() if l.strip()]
import shutil
out_dir = '/content/audio_out/batch'
os.makedirs(out_dir, exist_ok=True)

for i, line in enumerate(lines):
    try:
        print(f'[{i+1}/{len(lines)}] {line[:70]}{"..." if len(line) > 70 else ""}')
        path, sr = synth(
            line,
            audio_prompt_path=BATCH_AUDIO_PROMPT or None,
            max_tokens=BATCH_MAX_TOKENS,
            cfg_scale=BATCH_CFG_SCALE,
            temperature=BATCH_TEMPERATURE,
            top_p=BATCH_TOP_P,
            top_k=BATCH_TOP_K,
            seed=BATCH_SEED,
            out_name=f'{i:03d}.wav',
        )
        shutil.move(path, os.path.join(out_dir, f'{i:03d}.wav'))

    except Exception as e:
        print(f"  -> EXCEPTION: {e}")
        continue
print(f'\n[Done] {len(lines)} files written to {out_dir}')